<a href="https://colab.research.google.com/github/luca-fce/siniestros-futbol-caba/blob/main/notebook/3_1_siniestros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Limpieza de datos — Siniestros viales CABA (2021–2025)

**Pregunta del TP:** ¿Se registra un aumento significativo de siniestros viales en la Ciudad de Buenos Aires durante las 12 horas posteriores a la derrota de los equipos más populares de fútbol (Selección Argentina, River Plate y Boca Juniors)?

Este notebook toma las dos bases del Gobierno de la Ciudad (`siniestros_viales_hechos.csv` y `siniestros_viales_victimas.csv`), revisa su calidad, las limpia y deja solo los años **2021 a 2025**.

> Los datos se leen directamente desde el repositorio del grupo en GitHub, así el notebook corre en Colab sin subir archivos.

## 1. Carga de los datos

Los archivos vienen separados por `;` y con codificación UTF-8 con BOM, por eso usamos `sep=';'` y `encoding='utf-8-sig'`. Se leen todas las columnas como texto (`dtype=str`) para poder detectar valores raros (como `SD` o `#¡REF!`) antes de convertir tipos.

In [1]:
import pandas as pd
import numpy as np

URL = 'https://raw.githubusercontent.com/luca-fce/siniestros-futbol-caba/main/datos/Siniestros%20Viales%202019%20-%202025/'
hechos_raw = pd.read_csv(URL + 'siniestros_viales_hechos.csv', sep=';', encoding='utf-8-sig', dtype=str)
victimas_raw = pd.read_csv(URL + 'siniestros_viales_victimas.csv', sep=';', encoding='utf-8-sig', dtype=str)

print('Hechos:', hechos_raw.shape, '| Víctimas:', victimas_raw.shape)
hechos_raw.head()

Hechos: (65818, 21) | Víctimas: (75197, 9)


,id_siniestro,numero_total_de_victimas,numero_victimas_leve_siniestro,numero_victimas_grave_siniestro,numero_victimas_mortal_siniestro,fecha_siniestro,anio_siniestro,mes_siniestro,dia_siniestro,hora_siniestro,...,direccion_normalizada_siniestro,comuna_siniestro,tipo_de_via_siniestro,geocodificacion_plana,longitud_siniestro,latitud_siniestro,participantes_siniestro,modo_desplazamiento_victima,contraparte_siniestro,gravedad_siniestro
0,LC-2019-0000053,1,1,0,0,2019-01-01,2019,1,1,01:55:00,...,NaN,Comuna 8,AUTOPISTA,POINT (101813.84712503915943671 95578.55507230...,-58.44351,-34.669125,AUTO-UTILITARIO,AUTO,UTILITARIO,LEVE
1,LC-2019-0000647,1,0,1,0,2019-01-01,2019,1,1,18:50:00,...,DE LOS CONSTITUYENTES AV. y HABANA,Comuna 12,AVENIDA,POINT (97510.27137648081406951 105087.83369881...,-58.490436,-34.583403,MOTO-AUTO,MOTO,AUTO,GRAVE
2,LC-2019-0000445,3,3,0,0,2019-01-01,2019,1,1,13:40:00,...,NaN,Comuna 11,NaN,POINT (94371.44237885093025398 102884.19365105...,-58.524662,-34.603255,AUTO-AUTO,AUTO,AUTO,LEVE
3,LC-2019-0000194,1,1,0,0,2019-01-01,2019,1,1,07:15:00,...,NaN,Comuna 9,NaN,POINT (94030.76669932194636203 97681.071761248...,-58.528413,-34.650156,AUTO-CAMION,AUTO,CAMION,LEVE
4,LC-2019-0000329,1,1,0,0,2019-01-01,2019,1,1,12:24:00,...,NaN,Comuna 4,NaN,POINT (105367.86665769023238681 97877.75085908...,-58.404748,-34.648387,AUTO-MOVIL,AUTO,MOVIL,LEVE


In [2]:
victimas_raw.tail()

,id_siniestro,fecha_siniestro,anio_siniestro,modo_desplazamiento_victima,sexo_victima,edad_victima,GRAVEdad_victima,rol_victima,fecha_fallecimiento_victima
75192,LC-2025-0864021,2025-11-10,2025,UTILITARIO,SD,SD,LEVE,SD,NaN
75193,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75194,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75195,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75196,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Diagnóstico de calidad

Antes de limpiar, revisamos nulos, duplicados y la cantidad de registros por año.

In [3]:
print('Nulos en HECHOS:'); print(hechos_raw.isnull().sum(), '\n')
print('Nulos en VÍCTIMAS:'); print(victimas_raw.isnull().sum())

Nulos en HECHOS:
id_siniestro                            0
numero_total_de_victimas                0
numero_victimas_leve_siniestro          0
numero_victimas_grave_siniestro         0
numero_victimas_mortal_siniestro        0
fecha_siniestro                         0
anio_siniestro                          0
mes_siniestro                           0
dia_siniestro                           0
hora_siniestro                         77
rango_horario                          77
direccion_normalizada_siniestro     12900
comuna_siniestro                     3017
tipo_de_via_siniestro               12227
geocodificacion_plana                2367
longitud_siniestro                   2378
latitud_siniestro                    2378
participantes_siniestro                 0
modo_desplazamiento_victima             0
contraparte_siniestro                   0
gravedad_siniestro                      0
dtype: int64 

Nulos en VÍCTIMAS:
id_siniestro                       4
fecha_siniestro               

In [4]:
print('Duplicados exactos - hechos:', hechos_raw.duplicated().sum(), '| ids repetidos:', hechos_raw['id_siniestro'].duplicated().sum())
print('Duplicados exactos - víctimas:', victimas_raw.duplicated().sum())
pd.DataFrame({'hechos': hechos_raw['anio_siniestro'].value_counts(),
              'victimas': victimas_raw['anio_siniestro'].value_counts()}).sort_index()

Duplicados exactos - hechos: 0 | ids repetidos: 0


Duplicados exactos - víctimas: 1439


,hechos,victimas
anio_siniestro,,
2019,10155,11886
2020,6061,6936
2021,7818,9053
2022,9536,10647
2023,9927,11075
2024,10569,12481
2025,11752,13115


In [5]:
# Valores problemáticos detectados
print('hora_siniestro = SD:', (hechos_raw['hora_siniestro'] == 'SD').sum())
print('numero_total_de_victimas = SD:', (hechos_raw['numero_total_de_victimas'] == 'SD').sum())
print('Celdas con #¡REF! (error de Excel):', (hechos_raw == '#¡REF!').sum().sum())
print('Fechas sin cero a la izquierda (ej. 2025-1-29):', (~hechos_raw['fecha_siniestro'].str.match(r'^\d{4}-\d{2}-\d{2}$')).sum())
print('\nFormatos de comuna:', hechos_raw['comuna_siniestro'].dropna().unique()[:20])
print('\nMayúsculas mezcladas en contraparte:', sorted(hechos_raw['contraparte_siniestro'].unique()))

hora_siniestro = SD: 4
numero_total_de_victimas = SD: 3276
Celdas con #¡REF! (error de Excel): 5
Fechas sin cero a la izquierda (ej. 2025-1-29): 9406

Formatos de comuna: <StringArray>
[ 'Comuna 8', 'Comuna 12', 'Comuna 11',  'Comuna 9',  'Comuna 4',  'Comuna 3',
 'Comuna 14', 'Comuna 15',  'Comuna 1',  'Comuna 7',  'Comuna 2', 'Comuna 13',
  'Comuna 5',  'Comuna 6', 'Comuna 10',  'COMUNA 1',  'COMUNA 3', 'COMUNA 15',
        'SD',        '12']
Length: 20, dtype: str

Mayúsculas mezcladas en contraparte: ['#¡REF!', 'AUTO', 'BICICLETA', 'CAMION', 'CAMIONETA', 'MONOPATIN', 'MOTO', 'MOVIL', 'MULTIPLE', 'OBJETO FIJO', 'OTRO', 'PEATON', 'SD', 'TAXI', 'TRANSPORTE PUBLICO', 'TREN', 'UTILITARIO', 'auto', 'camion', 'moto', 'movil', 'peaton', 'taxi']


**Hallazgos principales:**

- **Hechos:** cada fila es un siniestro (no hay ids repetidos). Problemas: fechas de 2025 sin ceros a la izquierda, `SD` ("sin dato") en hora y en total de víctimas (todo 2025), una fila con `#¡REF!` (error de Excel en la carga original), comunas escritas como `"Comuna 1"` hasta 2024 y como `"1"` en 2025, y categorías con mayúsculas/minúsculas mezcladas (`auto` vs `AUTO`).
- **Víctimas:** 4 filas totalmente vacías al final del archivo, edades `SD`, mayúsculas mezcladas (`mortal`, `Conductor`) y una columna mal nombrada (`GRAVEdad_victima`).
- Los duplicados exactos de víctimas **no se eliminan**: un mismo siniestro puede tener dos víctimas con los mismos datos (ej. dos acompañantes sin edad registrada). De hecho, si se borran, la cantidad de víctimas coincide *menos* veces con el total informado en la tabla de hechos (95,0 % → 93,3 %).

## 3. Limpieza de la tabla de HECHOS

**Transformaciones y por qué:**

1. **Unificar valores faltantes:** `SD`, `#¡REF!` y vacíos pasan a `NaN`, para que pandas los trate como datos faltantes y no como una categoría más.
2. **Convertir tipos:** fechas a `datetime`, números a enteros, coordenadas a decimales.
3. **Crear `fecha_hora`** (fecha + hora del siniestro). Es la columna clave del TP: con ella se podrá contar cuántos siniestros caen dentro de las 12 h posteriores a cada partido.
4. **Normalizar textos:** mayúsculas y sin espacios sobrantes; comunas como número del 1 al 15.
5. **Recalcular el total de víctimas** cuando falta, como suma de leves + graves + mortales (en los datos completos esa suma coincide con el total en el 99,99 % de los casos).

In [6]:
h = hechos_raw.copy()

# 1. Valores faltantes unificados
h = h.replace({'SD': np.nan, '#¡REF!': np.nan, '': np.nan})
h['geocodificacion_plana'] = h['geocodificacion_plana'].replace('POINT (SD SD)', np.nan)

# 2. Tipos de dato
h['fecha_siniestro'] = pd.to_datetime(h['fecha_siniestro'], format='%Y-%m-%d')  # acepta '2025-1-9'
for c in ['numero_total_de_victimas', 'numero_victimas_leve_siniestro', 'numero_victimas_grave_siniestro',
          'numero_victimas_mortal_siniestro', 'anio_siniestro', 'mes_siniestro', 'dia_siniestro', 'rango_horario']:
    h[c] = pd.to_numeric(h[c], errors='coerce').astype('Int64')
for c in ['latitud_siniestro', 'longitud_siniestro']:
    h[c] = pd.to_numeric(h[c], errors='coerce')

# 5. Total de víctimas faltante = leves + graves + mortales
suma = h['numero_victimas_leve_siniestro'] + h['numero_victimas_grave_siniestro'] + h['numero_victimas_mortal_siniestro']
h['numero_total_de_victimas'] = h['numero_total_de_victimas'].fillna(suma)

# Año, mes y día se derivan de la fecha (había 1 fila inconsistente: fecha 28/11 y día 29)
h['anio_siniestro'] = h['fecha_siniestro'].dt.year
h['mes_siniestro'] = h['fecha_siniestro'].dt.month
h['dia_siniestro'] = h['fecha_siniestro'].dt.day

# 3. Fecha y hora juntas
h['fecha_hora'] = pd.to_datetime(h['fecha_siniestro'].dt.strftime('%Y-%m-%d') + ' ' + h['hora_siniestro'],
                                 format='%Y-%m-%d %H:%M:%S', errors='coerce')
h['tiene_hora'] = h['fecha_hora'].notna()
h['dia_semana'] = h['fecha_siniestro'].dt.dayofweek  # 0 = lunes ... 6 = domingo

# 4. Textos normalizados
texto = ['tipo_de_via_siniestro', 'participantes_siniestro', 'modo_desplazamiento_victima',
         'contraparte_siniestro', 'gravedad_siniestro', 'direccion_normalizada_siniestro']
for c in texto:
    h[c] = h[c].str.strip().str.upper()
h['comuna_siniestro'] = pd.to_numeric(h['comuna_siniestro'].str.extract(r'(\d+)')[0], errors='coerce').astype('Int64')
h = h.rename(columns={'comuna_siniestro': 'comuna'})

h.dtypes

id_siniestro                                   str
numero_total_de_victimas                     Int64
numero_victimas_leve_siniestro               Int64
numero_victimas_grave_siniestro              Int64
numero_victimas_mortal_siniestro             Int64
fecha_siniestro                     datetime64[us]
anio_siniestro                               int32
mes_siniestro                                int32
dia_siniestro                                int32
hora_siniestro                                 str
rango_horario                                Int64
direccion_normalizada_siniestro                str
comuna                                       Int64
tipo_de_via_siniestro                          str
geocodificacion_plana                          str
longitud_siniestro                         float64
latitud_siniestro                          float64
participantes_siniestro                        str
modo_desplazamiento_victima                    str
contraparte_siniestro          

## 4. Limpieza de la tabla de VÍCTIMAS

Se eliminan las filas vacías, se corrige el nombre de la columna de gravedad, se pasa la edad a número y se normalizan los textos.

In [7]:
v = victimas_raw.dropna(how='all').copy()
v = v.rename(columns={'GRAVEdad_victima': 'gravedad_victima'})
v = v.replace({'SD': np.nan, 'sd': np.nan, '': np.nan})

v['fecha_siniestro'] = pd.to_datetime(v['fecha_siniestro'], format='%Y-%m-%d')
v['fecha_fallecimiento_victima'] = pd.to_datetime(v['fecha_fallecimiento_victima'], format='%Y-%m-%d', errors='coerce')
v['anio_siniestro'] = v['fecha_siniestro'].dt.year
v['edad_victima'] = pd.to_numeric(v['edad_victima'], errors='coerce').astype('Int64')

for c in ['modo_desplazamiento_victima', 'sexo_victima', 'gravedad_victima', 'rol_victima']:
    v[c] = v[c].str.strip().str.upper()

print(v['gravedad_victima'].value_counts(dropna=False))
print(v['rol_victima'].value_counts(dropna=False))

gravedad_victima
LEVE      71308
GRAVE      3182
MORTAL      703
Name: count, dtype: int64
rol_victima
NaN          60136
PEATON        8405
CICLISTA      6281
CONDUCTOR      307
PASAJERO        64
Name: count, dtype: int64


## 5. Filtro 2021–2025

El período de análisis coincide con los partidos considerados (Selección Argentina, River y Boca, 2021–2025). Se descartan 2019 y 2020; además, 2020 estuvo muy afectado por la cuarentena, lo que hubiera distorsionado la comparación.

In [8]:
hechos = h[h['anio_siniestro'].between(2021, 2025)].sort_values('fecha_hora').reset_index(drop=True)
victimas = v[v['anio_siniestro'].between(2021, 2025)].reset_index(drop=True)

# Control: todas las víctimas deben corresponder a un hecho
print('Víctimas sin hecho asociado:', (~victimas['id_siniestro'].isin(hechos['id_siniestro'])).sum())
print('Hechos:', len(hechos), '| Víctimas:', len(victimas))
print('Hechos sin hora (no se pueden ubicar en la ventana de 12 h):', (~hechos['tiene_hora']).sum())
hechos.groupby('anio_siniestro').agg(siniestros=('id_siniestro', 'count'),
                                     victimas=('numero_total_de_victimas', 'sum'),
                                     mortales=('numero_victimas_mortal_siniestro', 'sum'))

Víctimas sin hecho asociado: 0
Hechos: 49602 | Víctimas: 56371
Hechos sin hora (no se pueden ubicar en la ventana de 12 h): 77


,siniestros,victimas,mortales
anio_siniestro,,,
2021,7818,9053,96
2022,9536,10647,111
2023,9927,11075,106
2024,10569,12481,113
2025,11752,13115,91


## 6. Serie horaria (base para el análisis de partidos)

Para comparar las 12 h posteriores a cada derrota contra períodos "normales", es útil tener la cantidad de siniestros **por hora**, incluyendo las horas con cero siniestros. Luego se cruza con la tabla de partidos (fecha y hora de finalización de cada partido + resultado).

In [9]:
siniestros_por_hora = (hechos.dropna(subset=['fecha_hora'])
                       .set_index('fecha_hora').resample('h').size()
                       .rename('siniestros').reset_index())
siniestros_por_hora['hora'] = siniestros_por_hora['fecha_hora'].dt.hour
siniestros_por_hora['dia_semana'] = siniestros_por_hora['fecha_hora'].dt.dayofweek
print(siniestros_por_hora.shape)
siniestros_por_hora.describe()

(43820, 4)


,fecha_hora,siniestros,hora,dia_semana
count,43820,43820.000000,43820.000000,43820.000000
mean,2023-07-02 21:30:00,1.130192,11.499087,3.000091
min,2021-01-01 00:00:00,0.000000,0.000000,0.000000
25%,2022-04-02 10:45:00,0.000000,5.000000,1.000000
50%,2023-07-02 21:30:00,1.000000,11.000000,3.000000
75%,2024-10-01 08:15:00,2.000000,17.000000,5.000000
max,2025-12-31 19:00:00,11.000000,23.000000,6.000000
std,NaN,1.311484,6.921914,2.000639


## 7. Exportación de las bases limpias

Se guardan en la carpeta de Colab; desde ahí se descargan y se suben al repo en `datos/Siniestros Viales 2021 - 2025 (limpios)/`.

In [10]:
hechos.to_csv('hechos_limpio_2021_2025.csv', index=False)
victimas.to_csv('victimas_limpio_2021_2025.csv', index=False)
siniestros_por_hora.to_csv('siniestros_por_hora_2021_2025.csv', index=False)
print('Archivos guardados.')

Archivos guardados.


## Próximo paso: cruzar con los partidos

1. Unificar los partidos (notebooks `3_1_seleccion` y `3_1_river_boca`) en una sola tabla con: `fecha_hora_fin`, `equipo`, `resultado` (G/E/P), `competencia`, `local/visitante`.
2. Para cada partido, sumar los siniestros en la ventana `[fecha_hora_fin, fecha_hora_fin + 12 h]`.
3. Comparar contra un **control**: la misma franja horaria y el mismo día de la semana en semanas sin partido (o sin derrota). Esto evita confundir el efecto del partido con el patrón normal (hay muchos más siniestros a las 17 h que a las 4 h, y los fines de semana son distintos).
4. Test estadístico: derrotas vs. victorias/empates vs. control (t de Student o Mann-Whitney; o una regresión de Poisson con controles de hora, día y año).